# 02 — Causal Attribution Labeling

Labels each post with 22 causal attribution categories using GPT-4o-mini.

**Input:** `reddit_estrangement_clean.csv`  
**Output:** `reddit_estrangement_labeled.csv`

> **Note:** Requires an OpenAI API key stored as `MY_TEST_KEY` in Colab secrets.

In [ ]:
!pip install openai --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Setup — load data and API client
import openai
import pandas as pd
from google.colab import userdata

client = openai.OpenAI(api_key=userdata.get("MY_TEST_KEY"))

df = pd.read_csv("/content/drive/MyDrive/Raw Data/reddit_estrangement_clean.csv")
print(f"Loaded: {len(df)} posts")

In [ ]:
# Codebook — 22 causal attribution categories
CODEBOOK = """
1 — Abuse (Şiddet/istismar)
Abuse is the improper, harmful, or cruel treatment of a person, often to exert power or control. It involves intentional actions – physical, emotional, sexual, or financial – that cause harm, distress, or injury.
Look for: Hitting, slapping, or kicking, name-calling, isolating an individual, or threatening behavior, any sexual act without consent.
NOT this code: Financial abuse/control over the child goes to Code 15.

2 — Family conflict/rivalry (Aile içi çatışma/rekabet)
Ongoing disputes, tension, or competition among family members. Parental favoritism, golden child vs scapegoat dynamics, unequal treatment.
Look for: Parents comparing children, parental attention favoring one sibling, feelings of being least favored.
NOT this code: Minor isolated day-to-day disagreements without systemic tension.

3 — Deceit/dishonesty (Kandırma/dürüst olmama)
Deliberately concealing truth, making false statements, or distorting reality to manipulate or control. Gaslighting, chronic lying, breaking significant promises, triangulation, financial deceit.
Look for: Gaslighting, chronic lying, breaking promises, triangulation, hiding major secrets, financial deceit.
NOT this code: Genuine forgetting, honest mistakes without pattern, different subjective memories without deliberate distortion.

4 — Divorce/remarriage (Boşanma/yeniden evlenme)
Familial disruption due to parents' separation or a parent's subsequent marriage. Parent prioritizing new partner, forcing child to choose sides, using child as weapon.
Look for: Parent abandoning child for new spouse, child used as messenger during divorce, pressure to hate other parent, child displaced by new family.
NOT this code: Divorce mentioned only as background timeline marker without being the source of conflict.

5 — Alcohol/substance use (Alkol/madde)
Parent's problematic recurring consumption of alcohol, drugs, or prescription medications severely disrupting family dynamic.
Look for: Parent frequently drunk or high, erratic behavior tied to intoxication, parent choosing substance over relationship, stealing from child to fund addiction.
NOT this code: Child's own substance abuse. Casual social drinking not cited as source of toxicity.

6 — Ingratitude/entitlement (Nankörlük/hak iddiası)
Parent's belief they are inherently owed the child's resources, time, emotional labor, or obedience. Parent demands things FROM the child.
Look for: Parent demanding money or gifts as their right, expecting child to drop everything for them, "I gave you life, you owe me" guilt trips, treating child as personal servant.
NOT this code: Parent demanding access to child's space or privacy (use 7d). Code 6 requires entitlement to resources or labor, not just physical/communicative access.

7a — Toxic-cruelty (Toksik-kırıcı/acımasız)
Deliberate, mean-spirited, ruthless behavior designed to inflict deep emotional pain, degrade, or humiliate. Malicious intent, profound lack of empathy. Chronic and continuous.
Look for: Exceptionally degrading statements, mocking child's trauma or appearance, intentional severe public humiliation, parent taking pleasure in child's suffering.
NOT this code: Explosive rage without targeted cruelty (use 7c). Dismissiveness without malice (use 7b). Physical harm (use 1).

7b — Toxic-disrespect (Toksik-saygısızlık)
Rude or discourteous behavior. Dismissiveness, passive-aggressive slights, invalidating child's opinions, talking down to child.
Look for: Talking to adult child like a child, passive-aggressive comments, publicly embarrassing behavior toward others, dismissing child's views.
NOT this code: Deliberate targeted cruelty (use 7a). Explosive anger (use 7c).

7c — Toxic-anger (Toksik-öfke)
Parent experiences anger quickly or often and expresses it in ways making child uncomfortable. Explosive, disproportionate reactions. Throwing objects, shouting, intimidation.
Look for: Child "walking on eggshells," avoiding going out in public with parent, disproportionate reactions to small annoyances.
NOT this code: Deliberate targeted cruelty (use 7a).

7d — Toxic-boundary violations (Toksik-sınır ihlali)
Repeating an action after child made clear they are uncomfortable. General lack of regard for privacy and personal space. Cannot take "No" for an answer.
Look for: Contacting on social media after being blocked, violating no-contact requests, circumventing restraining orders, entering bedroom without consent, reading diary.
NOT this code: General disrespect without specific boundary violations (use 7b).

7e — Toxic-generic (Toksik-genel)
Toxic behavior that doesn't fit neatly into 7a, 7b, 7c, or 7d.

8a — Voluntary physical distance (Gönüllü fiziksel uzaklık)
Voluntary long distance played a factor in estrangement. Parent had ability to live closer but chose not to, or adult child chose to live far away before estrangement.
Look for: Living far away making it hard to maintain relationship, feeling rejected by parent who didn't visit often.
NOT this code: Distance due to incarceration or involuntary commitment (use 8b).

8b — Involuntary physical distance (Zorunlu fiziksel uzaklık)
Involuntary long distance because parent was in prison or another institution without choice.
Look for: Parent in jail or prison during child's upbringing, parent in involuntary psychiatric commitment.

9a — Parent's partner (Ebeveynin partneri)
Parent has a partner who has unpleasant interactions with the child. Partner abusive toward child, parent defends partner's poor behavior, parent inappropriately involving child in dating life.
Look for: Partner insulting child, parent refusing to believe child about partner's behavior, parent prioritizing new partner over child.
NOT this code: Issue only with child's own partner. Step-parent who raised child from young age.

9b — Child's partner (Çocuğun partneri)
Parent refuses to accept child's romantic relationship. Disapproves due to race, gender, sexuality, social status. Claims partner turned child against parent.
Look for: Parent excluding child's partner from events, parent blaming partner for estrangement, constant criticism of child's partner.

10 — Mental health issues (Ruh sağlığı sorunları)
Parent's diagnosed or clearly observable psychiatric condition (depression, bipolar, schizophrenia, anxiety, PTSD) directly disrupting the relationship. Excludes personality disorders.
Look for: Parent hospitalized for psychiatric episode, paranoia or delusions affecting how parent treats child, severe depression causing neglect, erratic behavior attributed to untreated mental illness.
NOT this code: Personality disorders like narcissism, BPD, sociopathy (use 11 or relevant 7x code). Child's own mental health issues.

11 — Selfishness/egocentrism (Bencillik/benmerkezcilik)
Parent's pervasive self-centeredness — persistent pattern of prioritizing own needs to systematic exclusion of child's. Incapable of giving attention TO the child.
Look for: Parent making every situation about themselves, no genuine interest in child's life, responding to child's pain with "what about me?", unable to see child as separate person.
NOT this code: Parent demanding resources FROM child (use 6). Key distinction: Code 6 = parent extracts from child. Code 11 = parent cannot give to child.

12 — Disapproval/non-acceptance (Onaylanmama hissi)
Child's experience of being judged, criticized, or rejected for who they fundamentally are — identity, values, life choices. Conditional love: "I would love you IF you were different."
Look for: Parent disapproving of child's religion, sexuality, gender identity, career, partner, lifestyle. Child must perform false version of self for parental love.
NOT this code: If primary experience is being excluded from family events (use 13). Reserve 12 for rejection of fundamental identity.

13 — Exclusion/stigmatization (Dışlanma/damgalama)
Being actively ostracized within the family — scapegoated, excluded from events, spoken about negatively to others, labeled black sheep, family turned against the child.
Look for: Excluded from family gatherings or group chats, spoken about negatively to relatives, labeled "black sheep," collective shunning by broader family network.
NOT this code: Minor incidental exclusion without broader pattern. If primarily about identity disapproval (use 12).

14 — Emotional neglect (İlgisizlik/manevi destek görmeme)
Parent fails to respond with care or empathy when child is in genuine pain — specifically when parent was NOT the initial cause of that pain.
Look for: Parent failing to comfort child after loss or crisis, responding to distress with indifference, absent during health crisis, minimizing child's suffering ("you're overreacting").
NOT this code: Parent's dismissiveness in response to pain the parent themselves caused (code the original behavior). Generally cold household without specific incident (use 11).

15 — Material deprivation (Maddi yardımdan mahrum bırakılma)
Parent withholds financial support, resources, or economic security causing real hardship. Using money as control or punishment. Unfair inheritance.
Look for: Cutting off financial support as punishment, unfair inheritance distribution, refusing to provide basic needs despite having means, economic control to keep child dependent.
NOT this code: Parent genuinely unable (not unwilling) to provide. Parent demanding money FROM child (use 6).

16 — Unknown/unclear reason (Bilinmeyen/belirsiz sebep)
Estrangement occurred without child understanding why. Parent initiated estrangement with no explanation, or child genuinely cannot identify cause.
Look for: Parent suddenly went no-contact without explanation, child says they don't know why, child actively trying to make sense of the rupture.
NOT this code: Child understands reason but parent denies it. Post describes clear toxic behaviors leading to estrangement even if parent offers no acknowledgment.
"""

In [ ]:
# System prompt and label_post function
SYSTEM_PROMPT = f"""You are an expert coder for a research project on parental estrangement narratives from Reddit.

You will receive a Reddit post (title + body). Your task is to identify which estrangement causes are present in the post.

For each category below, output 1 (present) or 0 (absent). A post can have multiple 1s, or all 0s if no clear cause is mentioned.

Read the post carefully. Only code what is explicitly stated or strongly implied. Do not infer causes that are not mentioned.

{CODEBOOK}

CRITICAL DISTINCTIONS:
- Code 6 = parent demands resources FROM child. Code 15 = parent withholds resources FROM child. Direction matters.
- Code 11 = parent cannot give attention to child. Code 6 = parent demands things from child.
- Code 7a = malicious targeted cruelty. Code 7b = dismissive/disrespectful. Code 7c = explosive anger. Do not apply all three — pick the most fitting.
- Code 12 = disapproval of identity. Code 13 = active exclusion from family network. These can co-occur but are distinct.
- Code 14 = neglect when child is in pain from a cause the parent did NOT create.
- Code 16 ONLY when child genuinely does not know why estrangement occurred.

OUTPUT: Return only a valid JSON object with exactly these keys, no explanation, no markdown:
{{"1":0,"2":0,"3":0,"4":0,"5":0,"6":0,"7a":0,"7b":0,"7c":0,"7d":0,"7e":0,"8a":0,"8b":0,"9a":0,"9b":0,"10":0,"11":0,"12":0,"13":0,"14":0,"15":0,"16":0}}"""


import json
import time
from tqdm import tqdm

def label_post(title, text):
    content = f"TITLE: {title}\n\nPOST: {text[:2000]}"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content}
        ],
        temperature=0,
        max_tokens=200,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
# Test API connection
# Test API connection
test = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "hi"}],
    max_tokens=5
)
print("API connection OK:", test.choices[0].message.content)

In [ ]:
# Run labeling — parallel with resume support
from concurrent.futures import ThreadPoolExecutor, as_completed
import os, time, threading
from tqdm import tqdm

OUTPUT_PATH = "/content/drive/MyDrive/Raw Data/reddit_estrangement_labeled.csv"

# Resume from where we left off
try:
    df_done = pd.read_csv(OUTPUT_PATH)
    done_ids = set(df_done["post_id"].astype(str).tolist())
    print(f"Already labeled: {len(done_ids)} posts")
except FileNotFoundError:
    df_done = pd.DataFrame()
    done_ids = set()
    print("Starting fresh")

df_remaining = df[~df["post_id"].astype(str).isin(done_ids)].copy()
print(f"Remaining: {len(df_remaining)} posts")

SAVE_EVERY = 200
WORKERS = 3

results = []
errors = []
lock = threading.Lock()

def process_row(row):
    for attempt in range(5):
        try:
            result = label_post(row["title"], row["text"])
            result["post_id"] = str(row["post_id"])
            return result, None
        except Exception as e:
            wait = (attempt + 1) * 20
            time.sleep(wait)
    return None, {"post_id": str(row["post_id"])}

rows = [row for _, row in df_remaining.iterrows()]

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_row, row): row for row in rows}
    for i, future in enumerate(tqdm(as_completed(futures), total=len(rows))):
        result, error = future.result()
        if result:
            results.append(result)
        if error:
            errors.append(error)
        if (i + 1) % SAVE_EVERY == 0:
            with lock:
                df_batch = pd.DataFrame(results)
                df_all = pd.concat([df_done, df_batch], ignore_index=True)
                df_all.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
                df_done = df_all.copy()
                results = []
                print(f"\n✓ {len(df_done)} posts saved")

if results:
    df_batch = pd.DataFrame(results)
    df_all = pd.concat([df_done, df_batch], ignore_index=True)
    df_all.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print(f"\n✓ Done! Total: {len(df_all)} | Errors: {len(errors)}")